In [1]:
import math
import random
from dataclasses import dataclass, field
from typing import Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from draughts import Color, AlphaBetaEngine
from draughts.boards.standard import Board
from draughts.move import Move
from draughts.benchmark import Benchmark

In [2]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Neural Network Architecture (same as in ai.ipynb)

class ConvBlock(nn.Module):
    """Conv block with batch norm and residual connection."""
    def __init__(self, channels, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.dropout = nn.Dropout2d(dropout)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class PolicyValueNet(nn.Module):
    """CNN with policy and value heads."""
    def __init__(self, sq=50, channels=96, n_blocks=6, dropout=0.4):
        super().__init__()
        self.sq = sq
        
        self.input_conv = nn.Sequential(
            nn.Conv2d(4, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Dropout2d(0.1)
        )
        
        self.res_tower = nn.Sequential(*[ConvBlock(channels, dropout=0.15) for _ in range(n_blocks)])
        
        self.policy_conv = nn.Sequential(
            nn.Conv2d(channels, 32, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(dropout)
        )
        self.policy_fc = nn.Linear(32 * 10 * 5, sq * sq)
        
        self.value_conv = nn.Sequential(
            nn.Conv2d(channels, 4, 1),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Flatten()
        )
        self.value_fc = nn.Sequential(
            nn.Linear(4 * 10 * 5, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Tanh()
        )
    
    def _reshape_input(self, x):
        if x.dim() == 2:
            x = x.view(-1, 4, self.sq)
        return x.view(-1, 4, 10, 5)
    
    def forward(self, x):
        x = self._reshape_input(x)
        x = self.input_conv(x)
        x = self.res_tower(x)
        p = self.policy_conv(x)
        return self.policy_fc(p)
    
    def forward_both(self, x):
        """Returns (policy_logits, value) for MCTS."""
        x = self._reshape_input(x)
        x = self.input_conv(x)
        x = self.res_tower(x)
        
        p = self.policy_conv(x)
        policy = self.policy_fc(p)
        
        v = self.value_conv(x)
        value = self.value_fc(v)
        
        return policy, value

In [4]:
# Load trained model
net = PolicyValueNet(channels=96, n_blocks=6, dropout=0.4).to(device)
net.load_state_dict(torch.load("best_draughts_model.pt", map_location=device))
net.eval()
print(f"Model loaded: {sum(p.numel() for p in net.parameters()):,} parameters")

Model loaded: 5,021,521 parameters


## MCTS Node

Each node stores:
- Visit count and value sum for averaging
- Prior probability from the policy network
- Children dictionary mapping moves to child nodes

In [5]:
@dataclass
class MCTSNode:
    """A node in the MCTS tree."""
    board: Board
    parent: Optional["MCTSNode"] = None
    move: Optional[Move] = None  # Move that led to this node
    
    children: Dict[Move, "MCTSNode"] = field(default_factory=dict)
    visit_count: int = 0
    value_sum: float = 0.0
    prior: float = 0.0
    
    def is_expanded(self) -> bool:
        return len(self.children) > 0
    
    def is_terminal(self) -> bool:
        return self.board.game_over
    
    def value(self) -> float:
        """Average value of this node."""
        if self.visit_count == 0:
            return 0.0
        return self.value_sum / self.visit_count
    
    def ucb_score(self, c_puct: float) -> float:
        """Upper Confidence Bound for Trees (UCT) with policy prior."""
        if self.parent is None:
            return 0.0
        
        # Exploration term: prior * sqrt(parent_visits) / (1 + visits)
        exploration = c_puct * self.prior * math.sqrt(self.parent.visit_count) / (1 + self.visit_count)
        
        # Exploitation term: average value (negated because we want opponent's perspective)
        exploitation = -self.value()
        
        return exploitation + exploration
    
    def select_child(self, c_puct: float) -> "MCTSNode":
        """Select child with highest UCB score."""
        return max(self.children.values(), key=lambda c: c.ucb_score(c_puct))
    
    def expand(self, policy_probs: np.ndarray):
        """Create children for all legal moves with priors from policy."""
        for move in self.board.legal_moves:
            idx = self.board.move_to_index(move)
            child_board = self.board.copy()
            child_board.push(move)
            self.children[move] = MCTSNode(
                board=child_board,
                parent=self,
                move=move,
                prior=policy_probs[idx]
            )
    
    def backpropagate(self, value: float):
        """Backpropagate value up the tree."""
        node = self
        while node is not None:
            node.visit_count += 1
            node.value_sum += value
            value = -value  # Flip for opponent's perspective
            node = node.parent

## MCTS Engine

The engine runs MCTS simulations using:
1. **Selection**: Traverse tree using UCB until reaching a leaf
2. **Expansion**: Use policy network to get priors for all legal moves
3. **Evaluation**: Use value network to estimate position value
4. **Backpropagation**: Update visit counts and values up the tree

In [6]:
class MCTSEngine:
    """
    Simple AlphaZero-style MCTS engine with batched neural network evaluation.
    
    Uses neural network for:
    - Policy: Prior probabilities for move selection
    - Value: Leaf node evaluation
    """
    
    def __init__(self, net, simulations: int = 100, c_puct: float = 1.5, 
                 temperature: float = 0.0, batch_size: int = 8):
        """
        Args:
            net: PolicyValueNet model
            simulations: Number of MCTS simulations per move
            c_puct: Exploration constant (higher = more exploration)
            temperature: Move selection temperature (0 = greedy, higher = more random)
            batch_size: Number of positions to evaluate in parallel
        """
        self.net = net
        self.simulations = simulations
        self.c_puct = c_puct
        self.temperature = temperature
        self.batch_size = batch_size
        self.device = next(net.parameters()).device
        self.name = f"MCTS(sims={simulations})"
        self.nodes_searched = 0
    
    @torch.no_grad()
    def _evaluate(self, board: Board) -> tuple[np.ndarray, float]:
        """Get policy probabilities and value from neural network."""
        state = torch.from_numpy(board.to_tensor()).unsqueeze(0).to(self.device, dtype=torch.float32)
        policy_logits, value = self.net.forward_both(state)
        
        # Mask illegal moves and softmax
        mask = torch.from_numpy(board.legal_moves_mask()).to(self.device)
        policy_logits[0][~mask] = float('-inf')
        policy_probs = F.softmax(policy_logits[0], dim=0).cpu().numpy()
        
        # Negate value: NN outputs negative = good, but MCTS expects positive = good
        return policy_probs, -value.item()
    
    @torch.no_grad()
    def _evaluate_batch(self, boards: list[Board]) -> list[tuple[np.ndarray, float]]:
        """Batch evaluate multiple boards at once."""
        if not boards:
            return []
        
        # Stack all board tensors
        states = torch.stack([
            torch.from_numpy(b.to_tensor()) for b in boards
        ]).to(self.device, dtype=torch.float32)
        
        policy_logits, values = self.net.forward_both(states)
        
        results = []
        for i, board in enumerate(boards):
            # Mask illegal moves
            mask = torch.from_numpy(board.legal_moves_mask()).to(self.device)
            logits = policy_logits[i].clone()
            logits[~mask] = float('-inf')
            policy_probs = F.softmax(logits, dim=0).cpu().numpy()
            
            # Negate value: NN outputs negative = good, but MCTS expects positive = good
            results.append((policy_probs, -values[i].item()))
        
        return results
    
    def _select_leaf(self, root: MCTSNode) -> MCTSNode:
        """Select a leaf node using UCB."""
        node = root
        while node.is_expanded() and not node.is_terminal():
            node = node.select_child(self.c_puct)
        return node
    
    def _apply_virtual_loss(self, node: MCTSNode, amount: float = 1.0):
        """Apply virtual loss to discourage selecting same path again.
        
        UCB uses -value(), so to LOWER the score we need to INCREASE value_sum.
        """
        current = node
        sign = 1.0
        while current is not None:
            current.visit_count += 1
            current.value_sum += sign * amount  # Makes this path look worse
            sign = -sign  # Flip at each level
            current = current.parent
    
    def _revert_virtual_loss(self, node: MCTSNode, amount: float = 1.0):
        """Revert virtual loss before real backpropagation."""
        current = node
        sign = 1.0
        while current is not None:
            current.visit_count -= 1
            current.value_sum -= sign * amount
            sign = -sign
            current = current.parent
    
    def _search_batch(self, root: MCTSNode, batch_size: int) -> None:
        """Run a batch of MCTS simulations with batched NN evaluation."""
        leaves = []
        leaf_set = set()  # Track unique leaves by id
        
        # 1. Select multiple leaves, applying virtual loss
        for _ in range(batch_size):
            leaf = self._select_leaf(root)
            
            # Skip if already selected this exact leaf
            if id(leaf) in leaf_set:
                continue
            
            if leaf.is_terminal():
                # Handle terminal immediately
                if leaf.board.result == "1-0":
                    value = 1.0 if leaf.board.turn == Color.WHITE else -1.0
                elif leaf.board.result == "0-1":
                    value = 1.0 if leaf.board.turn == Color.BLACK else -1.0
                else:
                    value = 0.0
                leaf.backpropagate(value)
                self.nodes_searched += 1
                continue
            
            leaf_set.add(id(leaf))
            leaves.append(leaf)
            self._apply_virtual_loss(leaf)
        
        if not leaves:
            return
        
        # 2. Batch evaluate all leaves
        boards = [leaf.board for leaf in leaves]
        results = self._evaluate_batch(boards)
        
        # 3. Expand and backpropagate each leaf
        for leaf, (policy_probs, value) in zip(leaves, results):
            self._revert_virtual_loss(leaf)
            leaf.expand(policy_probs)
            leaf.backpropagate(value)
            self.nodes_searched += 1
    
    def get_best_move(self, board: Board, with_evaluation: bool = False):
        """Run MCTS and return the best move."""
        self.nodes_searched = 0
        
        # Create and expand root
        root = MCTSNode(board=board.copy())
        policy_probs, _ = self._evaluate(root.board)
        root.expand(policy_probs)
        root.visit_count = 1  # Root starts with 1 visit
        
        # Run simulations in batches
        remaining = self.simulations
        while remaining > 0:
            batch = min(self.batch_size, remaining)
            self._search_batch(root, batch)
            remaining -= batch
        
        # Select move by visit count
        if self.temperature == 0:
            best_move = max(root.children.items(), key=lambda x: x[1].visit_count)[0]
        else:
            moves = list(root.children.keys())
            visits = np.array([root.children[m].visit_count for m in moves], dtype=np.float64)
            visits = visits ** (1 / self.temperature)
            probs = visits / visits.sum()
            best_move = moves[np.random.choice(len(moves), p=probs)]
        
        if with_evaluation:
            # Child value is from opponent's perspective, negate for current player
            eval_score = -root.children[best_move].value() * 100
            return best_move, eval_score
        
        return best_move
    
    def get_move_probabilities(self, board: Board) -> dict[Move, float]:
        """Get visit count distribution after search."""
        root = MCTSNode(board=board.copy())
        policy_probs, _ = self._evaluate(root.board)
        root.expand(policy_probs)
        root.visit_count = 1
        
        remaining = self.simulations
        while remaining > 0:
            batch = min(self.batch_size, remaining)
            self._search_batch(root, batch)
            remaining -= batch
        
        total_visits = sum(c.visit_count for c in root.children.values())
        return {move: child.visit_count / total_visits for move, child in root.children.items()}

## Create Engine Instance

In [7]:
# Create MCTS engine with our trained network
engine = MCTSEngine(
    net=net,
    simulations=100,
    c_puct=1.5
)

print(f"Created {engine.name}")
print(f"  - Simulations: {engine.simulations}")
print(f"  - C_puct: {engine.c_puct}")
print(f"  - Temperature: {engine.temperature}")

Created MCTS(sims=100)
  - Simulations: 100
  - C_puct: 1.5
  - Temperature: 0.0


## Demo: Single Move

In [8]:
# Test on starting position
board = Board()
print("Starting position:")
print(board)
print()

move, score = engine.get_best_move(board, with_evaluation=True)
print(f"Best move: {move}")
print(f"Evaluation: {score:+.1f}")
print(f"Nodes searched: {engine.nodes_searched}")

Starting position:
. b . b . b . b . b     .  1 .  2 .  3 .  4 .  5
 b . b . b . b . b .      6 .  7 .  8 .  9 . 10 .
 . b . b . b . b . b     . 11 . 12 . 13 . 14 . 15
 b . b . b . b . b .     16 . 17 . 18 . 19 . 20 .
 . . . . . . . . . .     . 21 . 22 . 23 . 24 . 25
 . . . . . . . . . .     26 . 27 . 28 . 29 . 30 .
 . w . w . w . w . w     . 31 . 32 . 33 . 34 . 35
 w . w . w . w . w .     36 . 37 . 38 . 39 . 40 .
 . w . w . w . w . w     . 41 . 42 . 43 . 44 . 45
 w . w . w . w . w .     46 . 47 . 48 . 49 . 50 .

Best move: 35-30
Evaluation: -0.6
Nodes searched: 80


In [9]:
# Show move probabilities
probs = engine.get_move_probabilities(board)
print("Move probabilities (by visit count):")
for move, prob in sorted(probs.items(), key=lambda x: -x[1])[:5]:
    print(f"  {move}: {prob:.1%}")

Move probabilities (by visit count):
  35-30: 23.8%
  32-28: 18.8%
  31-26: 18.8%
  34-30: 12.5%
  33-28: 10.0%


## Demo: Play a Game

In [10]:
def play_game(engine1, engine2, max_moves=200, verbose=True):
    """Play a game between two engines."""
    board = Board()
    move_count = 0
    
    while not board.game_over and move_count < max_moves:
        current_engine = engine1 if board.turn == Color.WHITE else engine2
        move = current_engine.get_best_move(board)
        if isinstance(move, tuple):
            move = move[0]
        
        if verbose and move_count < 10:
            side = "White" if board.turn == Color.WHITE else "Black"
            print(f"{move_count+1}. {side}: {move}")
        
        board.push(move)
        move_count += 1
    
    if verbose:
        if move_count >= 10:
            print(f"... ({move_count} total moves)")
        print(f"\nResult: {board.result}")
    
    return board.result


# MCTS vs itself
print("MCTS vs MCTS (self-play):")
print("=" * 30)
result = play_game(engine, engine)

MCTS vs MCTS (self-play):
1. White: 35-30
2. Black: 20-25
3. White: 40-35
4. Black: 15-20
5. White: 45-40
6. Black: 10-15
7. White: 50-45
8. Black: 20-24
9. White: 33-29
10. Black: 24x33
... (136 total moves)

Result: 1/2-1/2


## Benchmark: MCTS vs AlphaBeta

In [11]:
# Compare MCTS vs AlphaBeta using the Benchmark module
# Lower c_puct = more exploitation of good moves (less exploration)
engine = MCTSEngine(
    net=net,
    simulations=800,
    c_puct=1.0  # Reduced from 1.5 for more exploitation
)

ab_engine = AlphaBetaEngine(depth_limit=3)
print(f"MCTS ({engine.simulations} sims, c_puct={engine.c_puct}) vs AlphaBeta (depth {ab_engine.depth_limit})")
print("=" * 50)
stats = Benchmark(engine, ab_engine, games=3).run()
print(stats)

ab_engine = AlphaBetaEngine(depth_limit=2)
print(f"\nMCTS vs AlphaBeta (depth {ab_engine.depth_limit})")
print("=" * 50)
stats = Benchmark(engine, ab_engine, games=3).run()
print(stats)

MCTS (800 sims, c_puct=1.0) vs AlphaBeta (depth 3)
Game 1/3: MCTS(sims=800) (113 moves)
Game 2/3: MCTS(sims=800) (49 moves)
Game 3/3: AlphaBetaEngine (61 moves)

  BENCHMARK: MCTS(sims=800) vs AlphaBetaEngine

  RESULTS: 2-1-0 (W-L-D)
  MCTS(sims=800) win rate: 66.7%
  Elo difference: +120

  PERFORMANCE
  Avg game length: 74.3 moves
  MCTS(sims=800): 819.8ms/move, 0 nodes/move
  AlphaBetaEngine: 10.3ms/move, 233 nodes/move
  Total time: 93.0s

  GAMES
    1. MCTS(sims=800)  (113 moves)
    2. MCTS(sims=800)  (49 moves)
    3. AlphaBetaEngine (61 moves)



MCTS vs AlphaBeta (depth 2)
Game 1/3: MCTS(sims=800) (121 moves)
Game 2/3: MCTS(sims=800) (47 moves)
Game 3/3: AlphaBetaEngine (37 moves)

  BENCHMARK: MCTS(sims=800) vs AlphaBetaEngine

  RESULTS: 2-1-0 (W-L-D)
  MCTS(sims=800) win rate: 66.7%
  Elo difference: +120

  PERFORMANCE
  Avg game length: 68.3 moves
  MCTS(sims=800): 685.6ms/move, 0 nodes/move
  AlphaBetaEngine: 2.9ms/move, 72 nodes/move
  Total time: 70.9s

  GAMES
    1

## Experiment: Different Simulation Counts

In [12]:
import time

print("Simulation count vs Speed:")
print("-" * 40)

board = Board()
for sims in [50, 100, 200, 400]:
    test_engine = MCTSEngine(net=net, simulations=sims, c_puct=1.5)
    
    start = time.perf_counter()
    move = test_engine.get_best_move(board)
    elapsed = time.perf_counter() - start
    
    print(f"Sims={sims:4d}: {elapsed:.3f}s, move={move}")

Simulation count vs Speed:
----------------------------------------
Sims=  50: 0.054s, move=34-30
Sims= 100: 0.084s, move=35-30
Sims= 200: 0.179s, move=31-26
Sims= 400: 0.348s, move=35-30


In [13]:
# Run benchmark between simulations=50 and simulations=200
mcts_50 = MCTSEngine(net=net, simulations=50, c_puct=1.5)
mcts_200 = MCTSEngine(net=net, simulations=200, c_puct=1.5)

print(f"{mcts_50.name} vs {mcts_200.name}")
print("=" * 50)

stats = Benchmark(mcts_50, mcts_200, games=10).run()
print(stats)

MCTS(sims=50) vs MCTS(sims=200)
Game 1/10: MCTS(sims=200) (122 moves)
Game 2/10: Draw (87 moves)
Game 3/10: MCTS(sims=200) (49 moves)
Game 4/10: Draw (81 moves)
Game 5/10: MCTS(sims=200) (119 moves)
Game 6/10: MCTS(sims=200) (52 moves)
Game 7/10: MCTS(sims=200) (100 moves)
Game 8/10: MCTS(sims=200) (60 moves)
Game 9/10: MCTS(sims=200) (36 moves)
Game 10/10: Draw (84 moves)

  BENCHMARK: MCTS(sims=50) vs MCTS(sims=200)

  RESULTS: 0-7-3 (W-L-D)
  MCTS(sims=50) win rate: 15.0%
  Elo difference: -301

  PERFORMANCE
  Avg game length: 79.0 moves
  MCTS(sims=50): 46.6ms/move, 0 nodes/move
  MCTS(sims=200): 181.8ms/move, 0 nodes/move
  Total time: 90.2s

  GAMES
    1. MCTS(sims=200)  (122 moves)
    2. Draw            (87 moves)
    3. MCTS(sims=200)  (49 moves)
    4. Draw            (81 moves)
    5. MCTS(sims=200)  (119 moves)
    6. MCTS(sims=200)  (52 moves)
    7. MCTS(sims=200)  (100 moves)
    8. MCTS(sims=200)  (60 moves)
    9. MCTS(sims=200)  (36 moves)
   10. Draw            (84